In [1]:
import pandas as pd
import os

In [2]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [3]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [4]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [5]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [6]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [7]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [8]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [9]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [10]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [11]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [12]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [13]:
df = df.fillna({
    'release_year': -1
})

In [14]:
df.shape

(49783, 18)

In [15]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [16]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                    10
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                    136
release_year                     0
genres_array                    77
production_countries_array     755
production_companies_array    2370
cast_array                     620
director_array                 151
writers_array                 2423
Name: empty_count, dtype: int64


In [17]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [18]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [19]:
df.to_csv(clean_path, index=False)

## ML

In [ ]:
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack, csr_matrix

In [22]:
def vectorize_list_column_top_n(df, col, top_n=500):
    # transformer les listes en chaînes
    texts = df[col].fillna("").apply(lambda x: " ".join(x) if isinstance(x, list) else "")
    vectorizer = CountVectorizer(max_features=top_n)
    return vectorizer.fit_transform(texts)

genres_vec = vectorize_list_column_top_n(df, "genres_array", top_n=20)  # genres peu nombreux
production_countries_vec = vectorize_list_column_top_n(df, "production_countries_array", top_n=50)  # pays de production
production_companies_vec = vectorize_list_column_top_n(df, "production_companies_array", top_n=100)  # entreprises de production
cast_vec = vectorize_list_column_top_n(df, "cast_array", top_n=500)     # top 500 acteurs
director_vec = vectorize_list_column_top_n(df, "director_array", top_n=200)  # top 200 réalisateurs
writers_vec = vectorize_list_column_top_n(df, "writers_array", top_n=200)  # top 200 scénaristes

In [ ]:
# # 1. Fonction de tokenisation rapide avec regex
# def tokenize_and_filter_regex(text):
#     tokens = re.findall(r"[a-zA-Z]+", str(text).lower())  # garde uniquement les mots alphabétiques
#     return [t for t in tokens if t not in ENGLISH_STOP_WORDS]  # enlève les stopwords sklearn

# # 2. Application sur tout le DataFrame
# df["overview_filtered"] = df["overview"].apply(tokenize_and_filter_regex)

In [ ]:
# # 3. Transformation en texte pour CountVectorizer
# df["overview_filtered_str"] = df["overview_filtered"].apply(lambda x: " ".join(x))

In [ ]:
# # 4. TF
# count_vectorizer = CountVectorizer(max_features=10000)
# overview_tf = count_vectorizer.fit_transform(df["overview_filtered_str"])

In [ ]:
# # 5. TF-IDF
# tfidf_transformer = TfidfTransformer()
# overview_tfidf = tfidf_transformer.fit_transform(overview_tf)

In [ ]:
# TF-IDF sur les descriptions
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
overview_tfidf = tfidf_vectorizer.fit_transform(df["overview"].fillna(""))

In [29]:
# -------------------------
# Scaling pour release_year
# -------------------------
release_year_values = df["release_year"].fillna(0).values.reshape(-1, 1)
scaler = StandardScaler()
release_year_scaled = scaler.fit_transform(release_year_values)
release_year_sparse = csr_matrix(release_year_scaled)

In [ ]:
# -------------------------
# Assemblage sparse
# -------------------------
X = hstack([overview_tfidf, release_year_sparse, genres_vec, cast_vec, director_vec, production_companies_vec, production_countries_vec, writers_vec])

In [34]:
# -------------------------
# 5. NearestNeighbors
# -------------------------
nn_model = NearestNeighbors(metric='cosine', algorithm='brute')
nn_model.fit(X)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [ ]:
# -------------------------
# 6. Fonction de recommandation
# -------------------------
def recommend(movie_idx, top_n=10):
    distances, indices = nn_model.kneighbors(X[movie_idx], n_neighbors=top_n+1)
    indices = indices.flatten()[1:]  # exclure le film lui-même
    return df.iloc[indices][['title', 'release_year']]

In [37]:
test = recommend(424)
test

,title,release_year
116,The Dark Knight,2008
264867,The Irishman,2019
5780,Elf,2003
33311,R.I.P.D.,2013
4566,Death Becomes Her,1992
632483,Outlaw Johnny Black,2023
4470,Eraser,1996
1261,Return of the Jedi,1983
110569,A Million Ways to Die in the West,2014
209663,Rogue One: A Star Wars Story,2016
